In [1]:
import pandas as pd
df = pd.read_csv("churn_data.csv")
print(df.shape)
print(df.describe().round(2))
print(df.isnull().sum())
print(df["churn"].value_counts())

(500, 6)
          age  salaire  anciennete  consommation   churn
count  500.00   500.00      500.00        500.00  500.00
mean    44.22  4666.31        7.14         48.66    0.04
std     15.04  1844.87        4.36         28.93    0.19
min     18.00  1209.00        0.00          0.00    0.00
25%     32.00  3124.75        3.00         23.75    0.00
50%     45.00  4679.00        7.00         47.00    0.00
75%     57.00  6225.25       11.00         74.25    0.00
max     69.00  7988.00       14.00         99.00    1.00
age             0
salaire         0
abonnement      0
anciennete      0
consommation    0
churn           0
dtype: int64
churn
0    481
1     19
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split
# 1. Conversion du texte de « abonnement » en nombres
df["abonnement"] = df["abonnement"].map({"Basique": 0, "Standard": 1, "Premium": 2})
# 2. Variables explicatives (features) et 3. variable cible (target)
X = df[["age", "salaire", "abonnement", "anciennete", "consommation"]]
y = df["churn"]
# 4. Séparation : 80 % pour entraîner, 20 % mis de côté pour tester
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)
print(X_train.shape)
print(X_test.shape)

(400, 5)
(100, 5)


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
# Une forêt de 100 arbres de décision qui votent ensemble.
# max_depth=5 limite la complexité de chaque arbre (évite l'apprentissage par cœur).
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
# .fit() déclenche l'APPRENTISSAGE : le modèle analyse X_train et y_train.
model.fit(X_train, y_train)
# Prédictions sur les données de test (jamais vues avant)
y_pred = model.predict(X_test)
# accuracy_score compare les prédictions à la vérité : pourcentage de bonnes réponses
print("Accuracy :", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy : 0.99
              precision    recall  f1-score   support

           0       0.99      1.00      0.99        95
           1       1.00      0.80      0.89         5

    accuracy                           0.99       100
   macro avg       0.99      0.90      0.94       100
weighted avg       0.99      0.99      0.99       100



In [4]:
import joblib
# joblib.dump() sérialise (transforme en fichier binaire) l'objet « model » entraîné.
joblib.dump(model, "model.pkl")
print("Modèle sauvegardé dans model.pkl")

Modèle sauvegardé dans model.pkl


In [5]:
# ---------- ESSAI 2 : un modèle plus simple ----------
model = RandomForestClassifier(n_estimators=50, max_depth=3, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 2 :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl") # écrase le fichier de l'Essai 1
# ---------- ESSAI 3 : on retire la colonne « salaire » ----------
X = df[["age", "abonnement", "anciennete", "consommation"]]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Accuracy Essai 3 (sans salaire) :", accuracy_score(y_test, y_pred))
joblib.dump(model, "model.pkl") # écrase le fichier de l'Essai 2
# ---------- Vérification : que contient VRAIMENT model.pkl ? ----------
loaded = joblib.load("model.pkl")
print("Nombre de features attendu :", loaded.n_features_in_)

Accuracy Essai 2 : 0.99
Accuracy Essai 3 (sans salaire) : 1.0
Nombre de features attendu : 4
